# P08 Enterprise DataOps Platform 源码全流程 Notebook

这本 notebook 不是通过 `subprocess` 调壳执行脚本，而是把 `project_8_dataops_platform` 里的完整流程源码直接展开到 notebook 中。

你可以把它当成一本按项目顺序组织的代码讲义来阅读：

- 主流程脚本按推荐执行顺序排在前面
- 辅助脚本和工具脚本排在后面
- 如果 README 里提到某个脚本但仓库里缺失，也会保留缺口说明

## 项目环境

- 项目目录：`project_8_dataops_platform`
- 建议 Conda 环境：`p8-dataops`

## 本 notebook 收录的源码文件顺序

1. `src/build_platform_specs.py` - 生成平台规格与治理设计 [存在]
2. `src/simulate_platform_ops.py` - 模拟平台运行 [存在]
3. `src/evaluate_platform.py` - 评估平台指标 [存在]
4. `src/render_p8_chapter.py` - 渲染章节预览（README 引用） [缺失]
5. `src/run_p8_checks.py` - 项目检查 [存在]
6. `src/pipeline_utils.py` - 辅助脚本 [存在]

## 关键产物

- `data/processed/platform_scope.json`
- `data/processed/architecture_spec.json`
- `data/processed/api_catalog.json`
- `data/processed/task_queues.json`
- `data/processed/governance_policy.json`
- `data/processed/operating_model.json`
- `data/processed/dataset_versions.jsonl`
- `data/processed/experiment_runs.jsonl`
- `data/processed/lineage_graph.json`
- `data/processed/rollback_events.jsonl`
- `data/processed/alerts.jsonl`
- `data/processed/audit_log.jsonl`
- `data/processed/incident_reviews.jsonl`
- `data/processed/sla_report.json`
- `data/console/ui_panels.json`
- `data/reports/p8_report.md`
- `data/reports/p8_chapter_preview.pdf`
- `data/reports/p8_preview_stats.json`
- `data/reports/p8_metrics.json`
- `data/reports/p8_test_results.json`
- `data/reports/p8_test_report.md`


## 项目 README

下面直接附上项目自带的 `README.md`，方便在 notebook 里连同源码一起看。

# P8 Enterprise DataOps Platform

This project builds a small, reproducible enterprise DataOps platform prototype covering scope definition, architecture, version and experiment lineage, observability, and governance.

## Scope

1. Platform goals and scope
   - Unified target for ingestion, processing, evaluation, release, monitoring, and audit.
   - Explicit tenant, project, role, and permission model.
2. Core architecture
   - Scheduler, metadata, storage, and service layers.
   - Platform APIs, task queues, and UI panels.
3. Version and experiment lineage
   - Dataset versions, experiment runs, lineage graph, and rollback records.
4. Observability and operations
   - Metrics, alerts, audit logs, SLA tracking, and incident reviews.
5. Organization and governance
   - Team interfaces, standard workflows, and exception handling.
6. Extension directions
   - Shared multi-BU platform, stronger policy automation, and disaster recovery.

## Environment

Dedicated conda environment:

```bash
conda activate p8-dataops
```

Environment files:

- `environment.yml`
- `environment.lock.yml`
- `environment.preview.yml`

Recommended creation commands:

```bash
conda env create -f environment.yml
conda env update -n p8-dataops -f environment.lock.yml --prune
conda env create -f environment.preview.yml
```

## Recommended Run Order

```bash
python src/build_platform_specs.py
python src/simulate_platform_ops.py
python src/evaluate_platform.py
python src/render_p8_chapter.py
python src/run_p8_checks.py
```

## Main Outputs

- `data/processed/platform_scope.json`
- `data/processed/architecture_spec.json`
- `data/processed/api_catalog.json`
- `data/processed/task_queues.json`
- `data/processed/governance_policy.json`
- `data/processed/operating_model.json`
- `data/processed/dataset_versions.jsonl`
- `data/processed/experiment_runs.jsonl`
- `data/processed/lineage_graph.json`
- `data/processed/rollback_events.jsonl`
- `data/processed/alerts.jsonl`
- `data/processed/audit_log.jsonl`
- `data/processed/incident_reviews.jsonl`
- `data/processed/sla_report.json`
- `data/console/ui_panels.json`
- `data/reports/p8_report.md`
- `data/reports/p8_chapter_preview.pdf`
- `data/reports/p8_preview_stats.json`
- `data/reports/p8_metrics.json`
- `data/reports/p8_test_results.json`
- `data/reports/p8_test_report.md`


## 源码目录概览

当前 `src/` 中实际存在的 Python 文件共 `5` 个：

- `src/build_platform_specs.py`
- `src/evaluate_platform.py`
- `src/pipeline_utils.py`
- `src/run_p8_checks.py`
- `src/simulate_platform_ops.py`


## 完整源码展开


### `src/build_platform_specs.py` - 生成平台规格与治理设计

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import CONSOLE_DIR, PROCESSED_DIR, ensure_standard_dirs, write_json

SCOPE_FILE = PROCESSED_DIR / "platform_scope.json"
ARCH_FILE = PROCESSED_DIR / "architecture_spec.json"
API_FILE = PROCESSED_DIR / "api_catalog.json"
QUEUE_FILE = PROCESSED_DIR / "task_queues.json"
GOVERNANCE_FILE = PROCESSED_DIR / "governance_policy.json"
OPERATING_MODEL_FILE = PROCESSED_DIR / "operating_model.json"
UI_FILE = CONSOLE_DIR / "ui_panels.json"


def build_scope() -> dict:
    roles = {
        "platform_admin": [
            "manage_tenants",
            "manage_roles",
            "approve_exceptions",
            "trigger_rollbacks",
            "view_audit_logs",
        ],
        "data_engineer": [
            "register_dataset",
            "launch_ingestion",
            "publish_data_version",
            "view_lineage",
        ],
        "ml_engineer": [
            "launch_experiment",
            "compare_runs",
            "register_model_version",
            "request_rollbacks",
        ],
        "reviewer": [
            "approve_release",
            "review_quality_gate",
            "annotate_incident_review",
        ],
        "analyst": [
            "view_reports",
            "query_metadata",
            "monitor_sla_dashboard",
        ],
    }
    return {
        "platform_goal": "Unify ingestion, processing, evaluation, versioning, observability, and governance for enterprise data projects.",
        "tenants": ["foundation-data", "legal-ai", "finance-ai"],
        "projects": [
            {"project_id": "mini_c4_pipeline", "owner_team": "foundation-data", "project_type": "corpus_build"},
            {"project_id": "legal_sft_factory", "owner_team": "legal-ai", "project_type": "sft_data"},
            {"project_id": "finance_rag_ops", "owner_team": "finance-ai", "project_type": "rag_eval"},
        ],
        "roles": roles,
        "security_boundaries": [
            "dataset publish requires approval and audit logging",
            "rollback requires privileged role or approved exception",
            "project isolation prevents cross-tenant writes",
        ],
    }


def build_architecture() -> dict:
    return {
        "layers": [
            {
                "name": "scheduler_layer",
                "responsibilities": ["cron orchestration", "event triggers", "retry policy", "dependency scheduling"],
            },
            {
                "name": "metadata_layer",
                "responsibilities": ["dataset registry", "experiment tracking", "lineage graph", "approval records"],
            },
            {
                "name": "storage_layer",
                "responsibilities": ["raw zone", "processed zone", "artifact store", "feature snapshots"],
            },
            {
                "name": "service_layer",
                "responsibilities": ["ingestion service", "evaluation service", "release gate", "notification service"],
            },
        ],
        "runtime_components": {
            "task_queue": ["ingest_queue", "eval_queue", "release_queue", "backfill_queue"],
            "api_gateway": ["dataset API", "experiment API", "lineage API", "incident API"],
            "ui_modules": ["project overview", "version compare", "lineage graph", "ops dashboard", "audit center"],
        },
        "deployment_model": {
            "control_plane": "metadata, approvals, and queue coordination",
            "data_plane": "batch jobs, evaluators, and artifact processing",
        },
    }


def build_api_catalog() -> list[dict]:
    return [
        {"name": "POST /datasets/register", "purpose": "register a dataset version draft", "auth_role": "data_engineer"},
        {"name": "POST /experiments/launch", "purpose": "launch an experiment against a dataset version", "auth_role": "ml_engineer"},
        {"name": "GET /lineage/{asset_id}", "purpose": "retrieve upstream and downstream lineage", "auth_role": "analyst"},
        {"name": "POST /releases/promote", "purpose": "promote an approved artifact to production", "auth_role": "reviewer"},
        {"name": "POST /rollbacks/trigger", "purpose": "rollback a regressed model or dataset", "auth_role": "platform_admin"},
        {"name": "GET /ops/sla", "purpose": "read SLA and incident summaries", "auth_role": "analyst"},
    ]


def build_queues() -> list[dict]:
    return [
        {"queue_name": "ingest_queue", "priority": "high", "used_for": ["raw sync", "schema validation"]},
        {"queue_name": "eval_queue", "priority": "high", "used_for": ["quality scoring", "smoke tests", "regression checks"]},
        {"queue_name": "release_queue", "priority": "medium", "used_for": ["approval checks", "promotion", "rollback prepare"]},
        {"queue_name": "backfill_queue", "priority": "low", "used_for": ["historical recompute", "late audit enrichment"]},
    ]


def build_governance() -> dict:
    return {
        "team_interfaces": [
            {"producer": "platform_team", "consumer": "business_ai_team", "contract": "shared APIs, queue SLA, on-call support"},
            {"producer": "data_engineering", "consumer": "review_ops", "contract": "quality gates, release checklist, rollback playbook"},
        ],
        "standard_workflows": [
            "dataset draft -> quality evaluation -> reviewer approval -> release promotion",
            "experiment launch -> metric compare -> regression review -> register or rollback",
            "alert fired -> incident triage -> owner assignment -> postmortem and exception closure",
        ],
        "exception_process": {
            "required_fields": ["exception_id", "reason", "risk_assessment", "approver", "expiry_date"],
            "example_cases": ["temporary SLA breach waiver", "emergency rollback approval", "schema migration bypass"],
        },
    }


def build_ui_panels() -> list[dict]:
    return [
        {"panel_id": "project_overview", "widgets": ["active projects", "latest releases", "pending approvals"]},
        {"panel_id": "version_compare", "widgets": ["dataset diff", "metric compare", "rollback button"]},
        {"panel_id": "lineage_view", "widgets": ["graph canvas", "upstream assets", "downstream releases"]},
        {"panel_id": "ops_dashboard", "widgets": ["queue lag", "SLA status", "alerts", "incident MTTR"]},
        {"panel_id": "audit_center", "widgets": ["approval log", "actor timeline", "exception tracker"]},
    ]


def build_operating_model() -> dict:
    return {
        "raci_matrix": [
            {
                "workstream": "scope_and_intake",
                "platform_team": "A",
                "data_engineering": "R",
                "ml_team": "C",
                "review_ops": "I",
                "security_compliance": "C",
            },
            {
                "workstream": "release_gate_and_quality",
                "platform_team": "A",
                "data_engineering": "C",
                "ml_team": "R",
                "review_ops": "R",
                "security_compliance": "I",
            },
            {
                "workstream": "lineage_and_metadata_freshness",
                "platform_team": "A",
                "data_engineering": "R",
                "ml_team": "C",
                "review_ops": "I",
                "security_compliance": "I",
            },
            {
                "workstream": "incident_response_and_rollback",
                "platform_team": "A",
                "data_engineering": "C",
                "ml_team": "R",
                "review_ops": "C",
                "security_compliance": "I",
            },
            {
                "workstream": "exception_review_and_audit",
                "platform_team": "R",
                "data_engineering": "I",
                "ml_team": "I",
                "review_ops": "C",
                "security_compliance": "A",
            },
        ],
        "oncall_rotation": [
            {
                "tier": "L1_platform_ops",
                "coverage": "09:00-21:00 UTC",
                "owner": "platform_team",
                "focus": ["scheduler health", "queue lag", "console availability"],
            },
            {
                "tier": "L2_service_owner",
                "coverage": "follow-the-sun",
                "owner": "project owners",
                "focus": ["release gate failures", "quality regression", "lineage gaps"],
            },
            {
                "tier": "L3_platform_admin",
                "coverage": "major incidents only",
                "owner": "platform_admin",
                "focus": ["tenant isolation", "privileged rollback", "policy override"],
            },
        ],
        "operating_cadence": [
            {
                "cadence": "daily",
                "meeting": "ops standup",
                "inputs": ["open alerts", "queue backlog", "pending approvals"],
                "outputs": ["owner assignment", "same-day mitigation"],
            },
            {
                "cadence": "weekly",
                "meeting": "release and SLA review",
                "inputs": ["release gate pass rate", "rollback events", "SLA report"],
                "outputs": ["capacity tuning", "policy updates", "weekly summary"],
            },
            {
                "cadence": "monthly",
                "meeting": "governance board",
                "inputs": ["audit exceptions", "tenant roadmap", "cost trend"],
                "outputs": ["budget decision", "roadmap reprioritization", "control backlog"],
            },
        ],
        "escalation_policy": {
            "sev1": "page L1 immediately, engage L2 within 10 minutes, platform_admin approves rollback or freeze.",
            "sev2": "assign owner during business hours, resolve within the same shift, review in weekly ops.",
            "sev3": "record in backlog and audit summary, close through standard workflow.",
        },
    }


def main() -> None:
    ensure_standard_dirs()
    scope = build_scope()
    architecture = build_architecture()
    apis = build_api_catalog()
    queues = build_queues()
    governance = build_governance()
    ui_panels = build_ui_panels()
    operating_model = build_operating_model()

    summary = {
        "tenant_count": len(scope["tenants"]),
        "project_count": len(scope["projects"]),
        "role_count": len(scope["roles"]),
        "layer_count": len(architecture["layers"]),
        "api_count": len(apis),
        "queue_count": len(queues),
        "ui_panel_count": len(ui_panels),
        "raci_workstream_count": len(operating_model["raci_matrix"]),
        "oncall_tier_count": len(operating_model["oncall_rotation"]),
        "cadence_count": len(operating_model["operating_cadence"]),
        "project_type_distribution": dict(Counter(item["project_type"] for item in scope["projects"])),
    }

    write_json(scope, SCOPE_FILE)
    write_json(architecture, ARCH_FILE)
    write_json(apis, API_FILE)
    write_json(queues, QUEUE_FILE)
    write_json(governance, GOVERNANCE_FILE)
    write_json(operating_model, OPERATING_MODEL_FILE)
    write_json(ui_panels, UI_FILE)
    print("✅ P8 平台范围、架构与治理规格生成完成。")
    print(summary)


if __name__ == "__main__":
    main()


### `src/simulate_platform_ops.py` - 模拟平台运行

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import PROCESSED_DIR, ensure_standard_dirs, write_json, write_jsonl

VERSIONS_FILE = PROCESSED_DIR / "dataset_versions.jsonl"
EXPERIMENTS_FILE = PROCESSED_DIR / "experiment_runs.jsonl"
LINEAGE_FILE = PROCESSED_DIR / "lineage_graph.json"
ROLLBACK_FILE = PROCESSED_DIR / "rollback_events.jsonl"
ALERTS_FILE = PROCESSED_DIR / "alerts.jsonl"
AUDIT_FILE = PROCESSED_DIR / "audit_log.jsonl"
INCIDENTS_FILE = PROCESSED_DIR / "incident_reviews.jsonl"
SLA_FILE = PROCESSED_DIR / "sla_report.json"
SUMMARY_FILE = PROCESSED_DIR / "ops_summary.json"


def build_dataset_versions() -> list[dict]:
    return [
        {
            "version_id": "mini_c4_v1",
            "project_id": "mini_c4_pipeline",
            "parent_version": None,
            "status": "released",
            "quality_score": 0.81,
            "record_count": 526,
            "owner": "foundation-data",
        },
        {
            "version_id": "mini_c4_v2",
            "project_id": "mini_c4_pipeline",
            "parent_version": "mini_c4_v1",
            "status": "released",
            "quality_score": 0.86,
            "record_count": 610,
            "owner": "foundation-data",
        },
        {
            "version_id": "legal_sft_v1",
            "project_id": "legal_sft_factory",
            "parent_version": None,
            "status": "released",
            "quality_score": 0.84,
            "record_count": 7737,
            "owner": "legal-ai",
        },
        {
            "version_id": "legal_sft_v2",
            "project_id": "legal_sft_factory",
            "parent_version": "legal_sft_v1",
            "status": "candidate",
            "quality_score": 0.82,
            "record_count": 8120,
            "owner": "legal-ai",
        },
        {
            "version_id": "finance_rag_v1",
            "project_id": "finance_rag_ops",
            "parent_version": None,
            "status": "released",
            "quality_score": 0.88,
            "record_count": 1341,
            "owner": "finance-ai",
        },
        {
            "version_id": "finance_rag_v2",
            "project_id": "finance_rag_ops",
            "parent_version": "finance_rag_v1",
            "status": "released",
            "quality_score": 0.91,
            "record_count": 1488,
            "owner": "finance-ai",
        },
    ]


def build_experiments() -> list[dict]:
    return [
        {
            "run_id": "exp_mini_c4_smoke_001",
            "project_id": "mini_c4_pipeline",
            "dataset_version": "mini_c4_v1",
            "artifact_id": "model_mini_001",
            "status": "completed",
            "metric_primary": 0.62,
            "latency_ms": 105,
            "release_decision": "accepted",
        },
        {
            "run_id": "exp_mini_c4_smoke_002",
            "project_id": "mini_c4_pipeline",
            "dataset_version": "mini_c4_v2",
            "artifact_id": "model_mini_002",
            "status": "completed",
            "metric_primary": 0.68,
            "latency_ms": 99,
            "release_decision": "accepted",
        },
        {
            "run_id": "exp_legal_lora_001",
            "project_id": "legal_sft_factory",
            "dataset_version": "legal_sft_v1",
            "artifact_id": "model_legal_001",
            "status": "completed",
            "metric_primary": 0.77,
            "latency_ms": 121,
            "release_decision": "accepted",
        },
        {
            "run_id": "exp_legal_lora_002",
            "project_id": "legal_sft_factory",
            "dataset_version": "legal_sft_v2",
            "artifact_id": "model_legal_002",
            "status": "regressed",
            "metric_primary": 0.72,
            "latency_ms": 145,
            "release_decision": "rolled_back",
        },
        {
            "run_id": "exp_finance_retriever_001",
            "project_id": "finance_rag_ops",
            "dataset_version": "finance_rag_v1",
            "artifact_id": "retriever_fin_001",
            "status": "completed",
            "metric_primary": 0.89,
            "latency_ms": 88,
            "release_decision": "accepted",
        },
        {
            "run_id": "exp_finance_retriever_002",
            "project_id": "finance_rag_ops",
            "dataset_version": "finance_rag_v2",
            "artifact_id": "retriever_fin_002",
            "status": "completed",
            "metric_primary": 0.93,
            "latency_ms": 81,
            "release_decision": "accepted",
        },
        {
            "run_id": "exp_finance_generator_003",
            "project_id": "finance_rag_ops",
            "dataset_version": "finance_rag_v2",
            "artifact_id": "generator_fin_003",
            "status": "failed",
            "metric_primary": 0.0,
            "latency_ms": 220,
            "release_decision": "retry_required",
        },
    ]


def build_rollbacks() -> list[dict]:
    return [
        {
            "rollback_id": "rb_legal_001",
            "trigger_asset": "model_legal_002",
            "fallback_asset": "model_legal_001",
            "reason": "faithfulness regression and latency increase",
            "approved_by": "platform_admin",
            "status": "completed",
        }
    ]


def build_alerts() -> list[dict]:
    return [
        {"alert_id": "alert_001", "severity": "high", "source": "release_gate", "message": "legal_sft_v2 quality regression", "status": "resolved"},
        {"alert_id": "alert_002", "severity": "medium", "source": "scheduler", "message": "finance generator queue latency spike", "status": "resolved"},
        {"alert_id": "alert_003", "severity": "low", "source": "audit", "message": "one approval note was missing reviewer comment", "status": "resolved"},
    ]


def build_audit_log() -> list[dict]:
    return [
        {"event_id": "audit_001", "actor": "data_engineer", "action": "publish_dataset", "target": "mini_c4_v2"},
        {"event_id": "audit_002", "actor": "ml_engineer", "action": "launch_experiment", "target": "exp_legal_lora_002"},
        {"event_id": "audit_003", "actor": "reviewer", "action": "reject_release", "target": "model_legal_002"},
        {"event_id": "audit_004", "actor": "platform_admin", "action": "trigger_rollback", "target": "rb_legal_001"},
        {"event_id": "audit_005", "actor": "analyst", "action": "view_sla_dashboard", "target": "finance_rag_ops"},
    ]


def build_incidents() -> list[dict]:
    return [
        {
            "incident_id": "inc_001",
            "project_id": "legal_sft_factory",
            "root_cause": "dataset refresh lowered answer faithfulness on validation",
            "mttr_minutes": 42,
            "follow_up": "tighten release gate on citation consistency",
        },
        {
            "incident_id": "inc_002",
            "project_id": "finance_rag_ops",
            "root_cause": "generator retry loop increased queue latency",
            "mttr_minutes": 31,
            "follow_up": "separate generator retries into backfill queue",
        },
    ]


def build_sla_report() -> dict:
    return {
        "dataset_publish_sla_hours": 6,
        "incident_response_sla_minutes": 30,
        "lineage_freshness_sla_minutes": 15,
        "measured": {
            "dataset_publish_p95_hours": 3.2,
            "incident_response_p95_minutes": 28,
            "lineage_freshness_p95_minutes": 9,
        },
        "status": {
            "dataset_publish": "met",
            "incident_response": "met",
            "lineage_freshness": "met",
        },
    }


def build_lineage(versions: list[dict], experiments: list[dict], rollbacks: list[dict]) -> dict:
    nodes = []
    edges = []
    for version in versions:
        nodes.append({"id": version["version_id"], "type": "dataset_version", "project_id": version["project_id"]})
        if version["parent_version"]:
            edges.append({"source": version["parent_version"], "target": version["version_id"], "relation": "derived_from"})
    for experiment in experiments:
        nodes.append({"id": experiment["run_id"], "type": "experiment_run", "project_id": experiment["project_id"]})
        nodes.append({"id": experiment["artifact_id"], "type": "artifact", "project_id": experiment["project_id"]})
        edges.append({"source": experiment["dataset_version"], "target": experiment["run_id"], "relation": "used_by"})
        edges.append({"source": experiment["run_id"], "target": experiment["artifact_id"], "relation": "produced"})
    for rollback in rollbacks:
        nodes.append({"id": rollback["rollback_id"], "type": "rollback_event", "project_id": "legal_sft_factory"})
        edges.append({"source": rollback["trigger_asset"], "target": rollback["rollback_id"], "relation": "rolled_back_by"})
        edges.append({"source": rollback["rollback_id"], "target": rollback["fallback_asset"], "relation": "restored"})
    return {"nodes": nodes, "edges": edges}


def main() -> None:
    ensure_standard_dirs()
    versions = build_dataset_versions()
    experiments = build_experiments()
    rollbacks = build_rollbacks()
    alerts = build_alerts()
    audit_log = build_audit_log()
    incidents = build_incidents()
    sla_report = build_sla_report()
    lineage = build_lineage(versions, experiments, rollbacks)

    summary = {
        "dataset_version_count": len(versions),
        "experiment_count": len(experiments),
        "rollback_count": len(rollbacks),
        "alert_count": len(alerts),
        "audit_event_count": len(audit_log),
        "incident_count": len(incidents),
        "released_version_count": sum(item["status"] == "released" for item in versions),
        "experiment_status_distribution": dict(Counter(item["status"] for item in experiments)),
    }

    write_jsonl(versions, VERSIONS_FILE)
    write_jsonl(experiments, EXPERIMENTS_FILE)
    write_json(lineage, LINEAGE_FILE)
    write_jsonl(rollbacks, ROLLBACK_FILE)
    write_jsonl(alerts, ALERTS_FILE)
    write_jsonl(audit_log, AUDIT_FILE)
    write_jsonl(incidents, INCIDENTS_FILE)
    write_json(sla_report, SLA_FILE)
    write_json(summary, SUMMARY_FILE)
    print("✅ P8 版本、实验、监控与治理链路模拟完成。")
    print(summary)


if __name__ == "__main__":
    main()


### `src/evaluate_platform.py` - 评估平台指标

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import PROCESSED_DIR, REPORTS_DIR, ensure_standard_dirs, load_json, load_jsonl, write_json

METRICS_FILE = REPORTS_DIR / "p8_metrics.json"
REPORT_FILE = REPORTS_DIR / "p8_report.md"


def main() -> None:
    ensure_standard_dirs()
    scope = load_json(PROCESSED_DIR / "platform_scope.json")
    architecture = load_json(PROCESSED_DIR / "architecture_spec.json")
    apis = load_json(PROCESSED_DIR / "api_catalog.json")
    queues = load_json(PROCESSED_DIR / "task_queues.json")
    governance = load_json(PROCESSED_DIR / "governance_policy.json")
    operating_model = load_json(PROCESSED_DIR / "operating_model.json")
    versions = load_jsonl(PROCESSED_DIR / "dataset_versions.jsonl")
    experiments = load_jsonl(PROCESSED_DIR / "experiment_runs.jsonl")
    lineages = load_json(PROCESSED_DIR / "lineage_graph.json")
    rollbacks = load_jsonl(PROCESSED_DIR / "rollback_events.jsonl")
    alerts = load_jsonl(PROCESSED_DIR / "alerts.jsonl")
    audit_log = load_jsonl(PROCESSED_DIR / "audit_log.jsonl")
    incidents = load_jsonl(PROCESSED_DIR / "incident_reviews.jsonl")
    sla_report = load_json(PROCESSED_DIR / "sla_report.json")
    ui_panels = load_json(PROCESSED_DIR.parent / "console" / "ui_panels.json")

    regressed_runs = [item for item in experiments if item["status"] == "regressed"]
    failed_runs = [item for item in experiments if item["status"] == "failed"]
    metrics = {
        "tenant_count": len(scope["tenants"]),
        "project_count": len(scope["projects"]),
        "role_count": len(scope["roles"]),
        "layer_count": len(architecture["layers"]),
        "api_count": len(apis),
        "queue_count": len(queues),
        "ui_panel_count": len(ui_panels),
        "dataset_version_count": len(versions),
        "released_version_count": sum(item["status"] == "released" for item in versions),
        "experiment_count": len(experiments),
        "experiment_status_distribution": dict(Counter(item["status"] for item in experiments)),
        "regression_run_count": len(regressed_runs),
        "failed_run_count": len(failed_runs),
        "rollback_count": len(rollbacks),
        "alert_count": len(alerts),
        "resolved_alert_rate": round(sum(item["status"] == "resolved" for item in alerts) / max(1, len(alerts)), 4),
        "audit_event_count": len(audit_log),
        "incident_count": len(incidents),
        "avg_incident_mttr_minutes": round(sum(item["mttr_minutes"] for item in incidents) / max(1, len(incidents)), 2),
        "sla_met_rate": round(sum(value == "met" for value in sla_report["status"].values()) / max(1, len(sla_report["status"])), 4),
        "lineage_node_count": len(lineages["nodes"]),
        "lineage_edge_count": len(lineages["edges"]),
        "governance_workflow_count": len(governance["standard_workflows"]),
        "raci_workstream_count": len(operating_model["raci_matrix"]),
        "oncall_tier_count": len(operating_model["oncall_rotation"]),
        "operating_cadence_count": len(operating_model["operating_cadence"]),
    }
    write_json(metrics, METRICS_FILE)

    report = f"""# P8 Enterprise DataOps Platform Report

## 1. 平台目标与范围

- 租户数：{metrics['tenant_count']}
- 项目数：{metrics['project_count']}
- 角色数：{metrics['role_count']}
- API 数：{metrics['api_count']}

## 2. 核心架构设计

- 核心层数：{metrics['layer_count']}
- 队列数：{metrics['queue_count']}
- UI 面板数：{metrics['ui_panel_count']}
- 血缘节点/边：{metrics['lineage_node_count']}/{metrics['lineage_edge_count']}

## 3. 版本与实验链路落地

- 数据版本数：{metrics['dataset_version_count']}
- 已发布版本数：{metrics['released_version_count']}
- 实验数：{metrics['experiment_count']}
- 实验状态分布：{metrics['experiment_status_distribution']}
- 回滚事件数：{metrics['rollback_count']}

## 4. 可观测性与运营体系

- 告警数：{metrics['alert_count']}
- 已解决告警比例：{metrics['resolved_alert_rate']}
- SLA 达标率：{metrics['sla_met_rate']}
- 平均事故恢复时长：{metrics['avg_incident_mttr_minutes']} 分钟

## 5. 组织协同与治理

- 审计事件数：{metrics['audit_event_count']}
- 事故复盘数：{metrics['incident_count']}
- 标准流程数：{metrics['governance_workflow_count']}
- RACI 工作流数：{metrics['raci_workstream_count']}
- 值班层级数：{metrics['oncall_tier_count']}
- 运营节奏数：{metrics['operating_cadence_count']}

## 6. 扩展方向

- 从单团队平台扩展到跨 BU 共享平台和统一资产目录。
- 引入实时特征、成本预算守卫和跨区域容灾切换。
- 将审批、例外申请和策略引擎进一步自动化。
"""
    REPORT_FILE.write_text(report, encoding="utf-8")
    print("✅ P8 报告生成完成。")
    print(report)


if __name__ == "__main__":
    main()


### `src/render_p8_chapter.py` - 渲染章节预览（README 引用）

仓库中当前没有找到这个文件，因此这里保留缺口说明，不生成代码单元。

> 说明：README 中列出了这一步，但当前 `src/` 目录里没有这个文件。notebook 会保留缺口说明。


### `src/run_p8_checks.py` - 项目检查

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import subprocess
from pathlib import Path

from pipeline_utils import CONSOLE_DIR, PROCESSED_DIR, REPORTS_DIR, ensure_standard_dirs, load_json, load_jsonl, write_json

ROOT_DIR = Path(__file__).resolve().parent.parent
SRC_DIR = ROOT_DIR / "src"
RESULTS_FILE = REPORTS_DIR / "p8_test_results.json"
REPORT_FILE = REPORTS_DIR / "p8_test_report.md"


def run_command(command: list[str], name: str) -> dict:
    result = subprocess.run(command, capture_output=True, text=True)
    return {
        "name": name,
        "command": command,
        "returncode": result.returncode,
        "passed": result.returncode == 0,
        "stdout": result.stdout.strip(),
        "stderr": result.stderr.strip(),
    }


def main() -> None:
    ensure_standard_dirs()
    scope = load_json(PROCESSED_DIR / "platform_scope.json")
    architecture = load_json(PROCESSED_DIR / "architecture_spec.json")
    apis = load_json(PROCESSED_DIR / "api_catalog.json")
    queues = load_json(PROCESSED_DIR / "task_queues.json")
    governance = load_json(PROCESSED_DIR / "governance_policy.json")
    operating_model = load_json(PROCESSED_DIR / "operating_model.json")
    versions = load_jsonl(PROCESSED_DIR / "dataset_versions.jsonl")
    experiments = load_jsonl(PROCESSED_DIR / "experiment_runs.jsonl")
    lineage = load_json(PROCESSED_DIR / "lineage_graph.json")
    rollbacks = load_jsonl(PROCESSED_DIR / "rollback_events.jsonl")
    alerts = load_jsonl(PROCESSED_DIR / "alerts.jsonl")
    audit_log = load_jsonl(PROCESSED_DIR / "audit_log.jsonl")
    incidents = load_jsonl(PROCESSED_DIR / "incident_reviews.jsonl")
    sla_report = load_json(PROCESSED_DIR / "sla_report.json")
    ui_panels = load_json(CONSOLE_DIR / "ui_panels.json")

    py_files = sorted(str(path) for path in SRC_DIR.glob("*.py"))
    command_checks = [
        run_command(["python", "-m", "py_compile", *py_files], "py_compile"),
        run_command(["python", str(SRC_DIR / "evaluate_platform.py")], "evaluate_platform"),
    ]

    dataset_checks = []
    dataset_checks.append(
        {
            "name": "required_files_exist",
            "passed": all(
                path.exists()
                for path in [
                    PROCESSED_DIR / "platform_scope.json",
                    PROCESSED_DIR / "architecture_spec.json",
                    PROCESSED_DIR / "api_catalog.json",
                    PROCESSED_DIR / "task_queues.json",
                    PROCESSED_DIR / "governance_policy.json",
                    PROCESSED_DIR / "operating_model.json",
                    PROCESSED_DIR / "dataset_versions.jsonl",
                    PROCESSED_DIR / "experiment_runs.jsonl",
                    PROCESSED_DIR / "lineage_graph.json",
                    PROCESSED_DIR / "alerts.jsonl",
                    PROCESSED_DIR / "audit_log.jsonl",
                    PROCESSED_DIR / "incident_reviews.jsonl",
                    PROCESSED_DIR / "sla_report.json",
                    CONSOLE_DIR / "ui_panels.json",
                ]
            ),
            "details": {},
        }
    )
    dataset_checks.append(
        {
            "name": "role_and_permission_model_present",
            "passed": len(scope["roles"]) >= 5 and all(len(perms) >= 3 for perms in scope["roles"].values()),
            "details": {"role_names": sorted(scope["roles"])},
        }
    )
    dataset_checks.append(
        {
            "name": "architecture_layers_complete",
            "passed": {"scheduler_layer", "metadata_layer", "storage_layer", "service_layer"} == {layer["name"] for layer in architecture["layers"]},
            "details": {"layer_names": [layer["name"] for layer in architecture["layers"]]},
        }
    )
    dataset_checks.append(
        {
            "name": "api_queue_ui_present",
            "passed": len(apis) >= 5 and len(queues) >= 4 and len(ui_panels) >= 5,
            "details": {"api_count": len(apis), "queue_count": len(queues), "ui_panel_count": len(ui_panels)},
        }
    )
    version_ids = {item["version_id"] for item in versions}
    dataset_checks.append(
        {
            "name": "version_lineage_links_valid",
            "passed": all(item["parent_version"] in version_ids or item["parent_version"] is None for item in versions),
            "details": {"version_count": len(versions)},
        }
    )
    experiment_dataset_refs_ok = all(item["dataset_version"] in version_ids for item in experiments)
    dataset_checks.append(
        {
            "name": "experiments_reference_versions",
            "passed": experiment_dataset_refs_ok,
            "details": {"experiment_count": len(experiments)},
        }
    )
    rolled_back_assets = {item["trigger_asset"] for item in rollbacks}
    regressed_assets = {item["artifact_id"] for item in experiments if item["status"] == "regressed"}
    dataset_checks.append(
        {
            "name": "regressions_have_rollback",
            "passed": regressed_assets.issubset(rolled_back_assets),
            "details": {"regressed_assets": sorted(regressed_assets), "rolled_back_assets": sorted(rolled_back_assets)},
        }
    )
    dataset_checks.append(
        {
            "name": "alerts_audit_incidents_present",
            "passed": len(alerts) >= 3 and len(audit_log) >= 5 and len(incidents) >= 2,
            "details": {"alert_count": len(alerts), "audit_count": len(audit_log), "incident_count": len(incidents)},
        }
    )
    dataset_checks.append(
        {
            "name": "sla_all_met",
            "passed": all(value == "met" for value in sla_report["status"].values()),
            "details": sla_report["status"],
        }
    )
    lineage_ids = {node["id"] for node in lineage["nodes"]}
    dataset_checks.append(
        {
            "name": "lineage_edges_resolve",
            "passed": all(edge["source"] in lineage_ids and edge["target"] in lineage_ids for edge in lineage["edges"]),
            "details": {"node_count": len(lineage["nodes"]), "edge_count": len(lineage["edges"])},
        }
    )
    dataset_checks.append(
        {
            "name": "governance_exception_process_present",
            "passed": "exception_process" in governance and len(governance["standard_workflows"]) >= 3,
            "details": {"workflow_count": len(governance["standard_workflows"])},
        }
    )
    dataset_checks.append(
        {
            "name": "operating_model_raci_complete",
            "passed": len(operating_model["raci_matrix"]) >= 5
            and all(
                {"workstream", "platform_team", "data_engineering", "ml_team", "review_ops", "security_compliance"}.issubset(item)
                for item in operating_model["raci_matrix"]
            ),
            "details": {"raci_workstream_count": len(operating_model["raci_matrix"])},
        }
    )
    dataset_checks.append(
        {
            "name": "operating_model_oncall_and_cadence_present",
            "passed": len(operating_model["oncall_rotation"]) >= 3 and len(operating_model["operating_cadence"]) >= 3,
            "details": {
                "oncall_tier_count": len(operating_model["oncall_rotation"]),
                "cadence_count": len(operating_model["operating_cadence"]),
            },
        }
    )

    overall_passed = all(item["passed"] for item in command_checks + dataset_checks)
    results = {
        "overall_passed": overall_passed,
        "total_checks": len(command_checks) + len(dataset_checks),
        "passed_checks": sum(item["passed"] for item in command_checks + dataset_checks),
        "command_checks": command_checks,
        "dataset_checks": dataset_checks,
    }
    write_json(results, RESULTS_FILE)

    lines = [
        "# P8 Test Report",
        "",
        f"- Overall status: {'PASS' if overall_passed else 'FAIL'}",
        f"- Passed checks: {results['passed_checks']}/{results['total_checks']}",
        "",
        "## Command Checks",
        "",
    ]
    for item in command_checks:
        lines.append(f"- {item['name']}: {'PASS' if item['passed'] else 'FAIL'}")
        lines.append(f"  - Command: `{' '.join(item['command'])}`")
    lines.extend(["", "## Dataset Checks", ""])
    for item in dataset_checks:
        lines.append(f"- {item['name']}: {'PASS' if item['passed'] else 'FAIL'}")
        lines.append(f"  - Details: `{item['details']}`")

    REPORT_FILE.write_text("\n".join(lines), encoding="utf-8")
    print("✅ P8 测试完成。")
    print(results)


if __name__ == "__main__":
    main()


### `src/pipeline_utils.py` - 辅助脚本

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Iterable

ROOT_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = DATA_DIR / "reports"
CONSOLE_DIR = DATA_DIR / "console"


def ensure_standard_dirs() -> None:
    for path in [DATA_DIR, PROCESSED_DIR, REPORTS_DIR, CONSOLE_DIR]:
        path.mkdir(parents=True, exist_ok=True)


def write_json(data: dict | list, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def load_json(path: Path) -> dict | list:
    return json.loads(path.read_text(encoding="utf-8"))


def write_jsonl(records: Iterable[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> list[dict]:
    records: list[dict] = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def deterministic_bucket(value: str, mod: int = 100) -> int:
    digest = hashlib.sha256(value.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % mod
